# 손 bbox 크롭 저장 — v2 (비대칭 마진)

v1 대비 변경점: **손끝 잘림 문제 해결**

| 항목 | v1 | v2 |
|---|---|---|
| 마진 | `MARGIN=0.0` 균등 | 좌우/위/아래 **비대칭** + 픽셀 하한 |
| 마진 결정 | 감으로 지정 | **Part 3에서 실측** 후 결정 |
| 전/후 비교 | 없음 | Part 4에서 잘림 심한 케이스 직접 비교 |
| 로그 | `crop_log.csv` | 마진 픽셀량(`pad_l/t/r/b`) 컬럼 추가 |

```
C:\Users\Win11Pro\Desktop\뼈데이터\전처리\train\
├── images\            # 크롭 결과 (원본 파일명 유지)
├── preview\           # 원본+박스 | 크롭 검수 이미지
├── crop_log_v2.csv    # bbox·마진·conf·QC 플래그
├── crop_failed_v2.csv # QC 플래그 케이스만
├── margin_measure.csv # Part 3 잘림량 실측 결과
├── missing_ids.txt    # 누락 ID
└── errors.txt         # 예외 목록
```

| Part | 내용 |
|---|---|
| 0 | 환경 + 경로 |
| 1 | 모델 로드 |
| 2 | 기본 설정 |
| 3 | **잘림량 실측** → 마진 권장값 산출 |
| 4 | **마진 확정** + 전/후 비교 |
| 5 | 예행 실행 6장 |
| 6 | 전체 배치 크롭 저장 |
| 7 | 통계 + 누락 대조 |
| 8 | QC 케이스 검수 |

**마진 설계 근거 2가지**
1. **픽셀 하한이 필요합니다.** 검출기의 회귀 오차는 박스 크기와 무관한 고정 픽셀 성격입니다. 비율만 쓰면 작은 박스에서 확장폭이 몇 px에 그쳐 정작 여유가 필요한 곳에 마진이 덜 붙습니다. → `max(비율, 최소px)`
2. **4면의 중요도가 다릅니다.** 위(손가락 끝)와 아래(요골 원위부)가 좌우보다 중요합니다. 특히 요골 원위부 성장판 손실은 MAE에 직접 반영됩니다.


## Part 0. 환경 + 경로 설정

In [ ]:
import sys, os, json, time, shutil
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm.auto import tqdm

# ─── 경로 ───────────────────────────────────────────────────────
SRC_DIR  = Path(r"C:\Users\Win11Pro\Desktop\뼈데이터\Bone+Age+Training+Set\boneage-training-dataset")
BASE_OUT = Path(r"C:\Users\Win11Pro\Desktop\뼈데이터\전처리\train")

CROP_DIR    = BASE_OUT / "images"
PREVIEW_DIR = BASE_OUT / "preview"
for d in (BASE_OUT, CROP_DIR, PREVIEW_DIR):
    d.mkdir(parents=True, exist_ok=True)

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

# ─── 한글 경로 안전 I/O ─────────────────────────────────────────
def imread_u(path, flags=cv2.IMREAD_GRAYSCALE):
    """cv2.imread는 Windows 비ASCII 경로에서 None 반환 후 무음 실패"""
    img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), flags)
    if img is None:
        raise IOError(f"디코드 실패: {path}")
    return img

def imwrite_u(path, img, quality=None):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    ext = path.suffix or ".png"
    params = [int(cv2.IMWRITE_JPEG_QUALITY), quality] if (quality and ext.lower() in (".jpg", ".jpeg")) else []
    ok, buf = cv2.imencode(ext, img, params)
    if not ok:
        raise IOError(f"인코딩 실패: {path}")
    buf.tofile(str(path))

def to_bgr(im):
    """YOLO predict는 3채널 HWC 전제 → 흑백 2D 변환"""
    return cv2.cvtColor(im, cv2.COLOR_GRAY2BGR) if im.ndim == 2 else im

# ─── 점검 ───────────────────────────────────────────────────────
print("Python:", sys.version.split()[0], "| Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))

assert SRC_DIR.exists(), f"원본 경로 없음: {SRC_DIR}"
FILES = sorted([p for p in SRC_DIR.rglob("*") if p.suffix.lower() in IMG_EXT])
assert FILES, f"이미지가 없습니다: {SRC_DIR}"

_t = imread_u(FILES[0])
print(f"\n원본 {len(FILES):,}장  ← {SRC_DIR}")
print(f"읽기 테스트: {FILES[0].name}  shape={_t.shape}  dtype={_t.dtype}")
print(f"저장 위치  : {CROP_DIR}")


## Part 1. 모델 로드

In [ ]:
MODEL_PT = None
# 예: MODEL_PT = Path(r"C:\work\hand_det\runs\hand_yolo26n_640\weights\best.pt")

SEARCH_ROOTS = [Path(r"C:\work\hand_det\runs"), BASE_OUT.parent, Path.cwd()]

if MODEL_PT is None:
    cands = [p for root in SEARCH_ROOTS if root.exists() for p in root.rglob("best.pt")]
    if not cands:
        raise FileNotFoundError(
            "best.pt를 찾지 못했습니다. MODEL_PT에 직접 경로를 넣으세요.\n"
            f"탐색 위치: {[str(r) for r in SEARCH_ROOTS]}")
    MODEL_PT = max(cands, key=lambda p: p.stat().st_mtime)

MODEL_PT = Path(MODEL_PT)
assert MODEL_PT.exists(), f"가중치 없음: {MODEL_PT}"

det = YOLO(str(MODEL_PT))
CLASS_NAMES = det.names

print("가중치   :", MODEL_PT)
print("수정시각 :", datetime.fromtimestamp(MODEL_PT.stat().st_mtime).strftime("%Y-%m-%d %H:%M"))
print("클래스   :", CLASS_NAMES)


## Part 2. 기본 설정

마진은 **Part 4에서 따로** 정합니다. 여기는 추론·저장 관련 설정만 둡니다.

| 설정 | 기본값 | 근거 |
|---|---|---|
| `CONF_TH` | 0.25 | 낮게 두고 QC로 거르는 편이 미검출보다 안전 |
| `NO_DET_POLICY` | `"full"` | 미검출 시 원본 전체 저장 + 플래그. `"skip"`이면 저장 안 함 |
| `OUT_SIZE` | `None` | 크롭 원본 해상도 유지. 리사이즈는 학습 스크립트에서 |
| `SAVE_EXT` | `".png"` | 무손실. 의료영상은 JPG 압축 아티팩트 회피가 원칙 |
| `SKIP_EXISTING` | True | 중단 후 재실행 시 이어서 진행 |


In [ ]:
# ═══════════════════════ 설정 ═══════════════════════
CONF_TH       = 0.25
IOU_TH        = 0.50
IMGSZ         = 640
BATCH         = 16
DEVICE        = 0 if torch.cuda.is_available() else "cpu"
USE_HALF      = torch.cuda.is_available()
READ_WORKERS  = 8

NO_DET_POLICY = "full"       # "full" | "skip"
OUT_SIZE      = None         # None=원본 크기 유지 / 512=고정
SQUARE_PAD    = False        # OUT_SIZE 지정 시 종횡비 유지 + 패딩
SAVE_EXT      = ".png"
JPG_QUALITY   = 95

SAVE_PREVIEW  = "flagged"    # "all" | "flagged" | "none"
PREVIEW_MAX_H = 900

SKIP_EXISTING = True
LIMIT         = None         # 예행: 50
# ── QC 임계값 ──
LOW_CONF, MIN_AREA, MAX_AREA = 0.60, 0.10, 0.95
ASPECT_LO, ASPECT_HI = 0.8, 3.0
# ════════════════════════════════════════════════════

def qc_check(cf, n_box, x1, y1, x2, y2, W, H):
    flags = []
    bw, bh = x2 - x1, y2 - y1
    area_ratio = (bw * bh) / (W * H)
    aspect = bh / max(bw, 1)
    if n_box > 1:                              flags.append("multi_det")
    if cf < LOW_CONF:                          flags.append("low_conf")
    if area_ratio < MIN_AREA:                  flags.append("small_box")
    if area_ratio > MAX_AREA:                  flags.append("huge_box")
    if not (ASPECT_LO <= aspect <= ASPECT_HI): flags.append("odd_aspect")
    if x1 <= 1 or y1 <= 1 or x2 >= W - 1 or y2 >= H - 1: flags.append("touch_border")
    return flags, round(area_ratio, 5), round(aspect, 4)

def parse_boxes(result):
    """YOLO 결과 → confidence 내림차순 [(xyxy, conf, cls), ...]"""
    if result.boxes is None or len(result.boxes) == 0:
        return []
    xyxy  = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    clss  = result.boxes.cls.cpu().numpy().astype(int)
    order = np.argsort(-confs)
    return [(xyxy[i].tolist(), float(confs[i]), int(clss[i])) for i in order]

def finalize(crop):
    if OUT_SIZE is None:
        return crop
    h, w = crop.shape[:2]
    if not SQUARE_PAD:
        return cv2.resize(crop, (OUT_SIZE, OUT_SIZE), interpolation=cv2.INTER_AREA)
    s = OUT_SIZE / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    r = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((OUT_SIZE, OUT_SIZE), dtype=crop.dtype)
    top, left = (OUT_SIZE - nh) // 2, (OUT_SIZE - nw) // 2
    canvas[top:top + nh, left:left + nw] = r
    return canvas

print(f"conf={CONF_TH} out_size={OUT_SIZE} ext={SAVE_EXT} no_det={NO_DET_POLICY} "
      f"batch={BATCH} half={USE_HALF}")


## Part 3. 잘림량 실측 → 마진 권장값 산출

**감으로 마진을 정하지 않습니다.**

샘플 200장에서 검출 박스 주변을 넉넉히(15%) 넓힌 뒤 Otsu + 최대 연결성분으로
**실제 손 윤곽**을 구하고, 검출 박스 4변이 그 윤곽보다 몇 px 안쪽인지 잽니다.
양수 = 그만큼 잘렸다는 뜻입니다.

측정 결과의 p95에 안전계수 1.3을 곱한 값이 권장 마진입니다.
(1~2분 소요. 이미 값을 알고 있다면 건너뛰고 Part 4에서 직접 입력해도 됩니다.)


In [ ]:
MEASURE_N    = 200      # 샘플 수
PROBE_MARGIN = 0.15     # 측정용 탐색 여유 (최종 마진과 무관)

rng = np.random.default_rng(42)
sample = [FILES[i] for i in rng.choice(len(FILES), size=min(MEASURE_N, len(FILES)), replace=False)]

def hand_extent(img, box, probe=PROBE_MARGIN):
    """박스 주변을 넓게 보고 손의 실제 tight bbox 추정 → (x1,y1,x2,y2) or None"""
    H, W = img.shape[:2]
    x1, y1, x2, y2 = box
    mw, mh = (x2 - x1) * probe, (y2 - y1) * probe
    px1, py1 = int(max(0, x1 - mw)), int(max(0, y1 - mh))
    px2, py2 = int(min(W, x2 + mw)), int(min(H, y2 + mh))
    roi = img[py1:py2, px1:px2]
    if roi.size == 0:
        return None
    blur = cv2.GaussianBlur(roi, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN,
                          cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(th, 8)
    if n <= 1:
        return None
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    sx, sy, sw, sh, _ = stats[k]
    return (px1 + sx, py1 + sy, px1 + sx + sw, py1 + sy + sh)

rows = []
for i in tqdm(range(0, len(sample), BATCH), desc="measure"):
    chunk, imgs, keep = sample[i:i + BATCH], [], []
    for p in chunk:
        try:
            imgs.append(imread_u(p)); keep.append(p)
        except Exception:
            pass
    if not imgs:
        continue
    res = det.predict([to_bgr(im) for im in imgs], conf=CONF_TH, iou=IOU_TH, imgsz=IMGSZ,
                      device=DEVICE, half=USE_HALF, verbose=False, max_det=10)
    for p, im, r in zip(keep, imgs, res):
        bxs = parse_boxes(r)
        if not bxs:
            continue
        (bx1, by1, bx2, by2), cf, _ = bxs[0]
        ext = hand_extent(im, (bx1, by1, bx2, by2))
        if ext is None:
            continue
        ex1, ey1, ex2, ey2 = ext
        rows.append(dict(
            image_id=p.stem, conf=round(cf, 4),
            box_w=bx2 - bx1, box_h=by2 - by1,
            clip_left=bx1 - ex1, clip_right=ex2 - bx2,      # 양수 = 잘림
            clip_top=by1 - ey1,  clip_bottom=ey2 - by2,
        ))

mdf = pd.DataFrame(rows)
mdf.to_csv(BASE_OUT / "margin_measure.csv", index=False, encoding="utf-8-sig")
SIDES = ["clip_left", "clip_right", "clip_top", "clip_bottom"]

print(f"측정 성공 {len(mdf)}/{len(sample)}장\n")
print("[4변 잘림량 (px, 양수 = 검출박스가 손보다 안쪽)]")
display(mdf[SIDES].describe(percentiles=[.5, .9, .95, .99]).round(1))

def _p95_frac(side, dim):
    q = max(0.0, mdf[side].quantile(0.95))
    return q, q / mdf[dim].median()

print("\n[p95 기준]")
for s, d in [("clip_left", "box_w"), ("clip_right", "box_w"),
             ("clip_top", "box_h"), ("clip_bottom", "box_h")]:
    q, f = _p95_frac(s, d)
    print(f"  {s:12s} {q:6.1f}px   비율 {f:.4f}")

SF = 1.3   # 안전계수
REC_X   = max(0.01, round(max(_p95_frac("clip_left",  "box_w")[1],
                              _p95_frac("clip_right", "box_w")[1]) * SF, 3))
REC_TOP = max(0.01, round(_p95_frac("clip_top",    "box_h")[1] * SF, 3))
REC_BOT = max(0.01, round(_p95_frac("clip_bottom", "box_h")[1] * SF, 3))
REC_PX  = int(np.clip(np.ceil(mdf[SIDES].quantile(0.95).max()), 4, 40))

print(f"\n→ Part 4 권장값 (안전계수 {SF})")
print(f"   MARGIN_X      = {REC_X}")
print(f"   MARGIN_TOP    = {REC_TOP}")
print(f"   MARGIN_BOTTOM = {REC_BOT}")
print(f"   MARGIN_MIN_PX = {REC_PX}")

fig, ax = plt.subplots(1, 4, figsize=(19, 3.4))
for a, s in zip(ax, SIDES):
    a.hist(mdf[s], bins=40, color="#3b7dd8")
    a.axvline(0, c="k", lw=1)
    a.axvline(mdf[s].quantile(0.95), c="r", ls="--", label="p95")
    a.set_title(s); a.legend()
plt.tight_layout(); plt.show()


## Part 4. 마진 확정 + 전/후 비교

Part 3 권장값을 아래에 반영하세요. `USE_MEASURED = True`면 자동 반영됩니다.

**설계**
- `max(비율, 최소px)` — 작은 박스에서도 최소 확장폭 보장
- 위(손가락 끝) > 아래(요골 원위부) > 좌우 순으로 여유
- 이미지 경계는 자동 클리핑 → 손이 잘려 촬영된 경우도 안전


In [ ]:
USE_MEASURED = True    # Part 3 측정값 자동 반영. False면 아래 수동값 사용

# ═══════════ 마진 (비대칭 + 픽셀 하한) ═══════════
MARGIN_X      = 0.020   # 좌우 (박스 폭 대비)   — 엄지 바깥쪽
MARGIN_TOP    = 0.045   # 위   (박스 높이 대비) — 손가락 끝
MARGIN_BOTTOM = 0.030   # 아래 (박스 높이 대비) — 손목·요골 원위부
MARGIN_MIN_PX = 8       # 비율과 무관한 최소 확장 px
MARGIN_MAX_PX = 120     # 폭주 방지 상한
# ════════════════════════════════════════════════

if USE_MEASURED and "REC_X" in dir():
    MARGIN_X, MARGIN_TOP, MARGIN_BOTTOM, MARGIN_MIN_PX = REC_X, REC_TOP, REC_BOT, REC_PX
    print("Part 3 측정값 적용")

def expand_box(x1, y1, x2, y2, W, H):
    """
    검출 박스를 비대칭 확장.
    검출기 회귀 오차는 박스 크기와 무관한 고정 px 성격이므로
    비율만 쓰면 작은 박스에서 확장폭이 부족해진다 → max(비율, 최소px).
    반환: (x1, y1, x2, y2, pad_l, pad_t, pad_r, pad_b)
    """
    bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
    pl = pr = int(np.clip(max(bw * MARGIN_X,      MARGIN_MIN_PX), 0, MARGIN_MAX_PX))
    pt      = int(np.clip(max(bh * MARGIN_TOP,    MARGIN_MIN_PX), 0, MARGIN_MAX_PX))
    pb      = int(np.clip(max(bh * MARGIN_BOTTOM, MARGIN_MIN_PX), 0, MARGIN_MAX_PX))
    nx1, ny1 = int(max(0, x1 - pl)), int(max(0, y1 - pt))
    nx2, ny2 = int(min(W, x2 + pr)), int(min(H, y2 + pb))
    return nx1, ny1, nx2, ny2, pl, pt, pr, pb

MARGIN_CFG = dict(x=MARGIN_X, top=MARGIN_TOP, bottom=MARGIN_BOTTOM,
                  min_px=MARGIN_MIN_PX, max_px=MARGIN_MAX_PX)
print("마진:", MARGIN_CFG)

# ─── 전/후 비교: 잘림이 심했던 케이스 우선 ──────────────────────
COMPARE_N = 4
if "mdf" in dir() and len(mdf):
    worst = (mdf.assign(w=mdf[SIDES].max(axis=1))
                .sort_values("w", ascending=False).head(COMPARE_N).image_id.tolist())
    picks = [p for p in FILES if p.stem in worst] or FILES[:COMPARE_N]
else:
    picks = FILES[:COMPARE_N]

fig, axes = plt.subplots(2, len(picks), figsize=(3.3 * len(picks), 9.5))
for j, p in enumerate(picks):
    im = imread_u(p); H, W = im.shape[:2]
    r = det.predict(to_bgr(im), conf=CONF_TH, iou=IOU_TH, imgsz=IMGSZ,
                    device=DEVICE, half=USE_HALF, verbose=False, max_det=10)[0]
    bxs = parse_boxes(r)
    if not bxs:
        continue
    (bx1, by1, bx2, by2), cf, _ = bxs[0]
    old = im[int(by1):int(by2), int(bx1):int(bx2)]
    nx1, ny1, nx2, ny2, pl, pt, pr, pb = expand_box(bx1, by1, bx2, by2, W, H)
    new = im[ny1:ny2, nx1:nx2]

    axes[0, j].imshow(old, cmap="gray")
    axes[0, j].set_title(f"{p.stem} 마진 없음\n{old.shape[1]}x{old.shape[0]}", fontsize=9)
    axes[1, j].imshow(new, cmap="gray")
    axes[1, j].set_title(f"마진 적용 +L{pl}/T{pt}/R{pr}/B{pb}\n{new.shape[1]}x{new.shape[0]}",
                         fontsize=9, color="green")
    axes[0, j].axis("off"); axes[1, j].axis("off")
plt.suptitle("위: 마진 없음 / 아래: 비대칭 마진 적용")
plt.tight_layout(); plt.show()


## Part 5. 예행 실행 6장

저장될 크롭을 최종 확인합니다. 손가락 끝과 손목 원위부가 모두 들어왔는지 보세요.
여전히 닿으면 Part 4로 돌아가 `MARGIN_TOP`을 0.01씩 올립니다.

In [ ]:
def crop_from(img, result, name=""):
    """(크롭, 기록 dict). 미검출 시 원본 전체 + no_det 플래그."""
    H, W = img.shape[:2]
    boxes = parse_boxes(result)
    rec = dict(image_id=Path(name).stem, file_name=name, W=W, H=H, num_boxes=len(boxes))

    if not boxes:
        # 미검출은 no_det 하나만 남긴다 (원본 전체가 fallback이라
        # huge_box/touch_border가 같이 붙으면 실제 원인이 묻힌다)
        rec.update(conf=0.0, x1=0, y1=0, x2=W, y2=H, crop_w=W, crop_h=H,
                   pad_l=0, pad_t=0, pad_r=0, pad_b=0,
                   area_ratio=1.0, aspect=round(H / max(W, 1), 4),
                   flags="no_det", saved=False)
        return img, rec

    (bx1, by1, bx2, by2), cf, cid = boxes[0]
    x1, y1, x2, y2, pl, pt, pr, pb = expand_box(bx1, by1, bx2, by2, W, H)
    flags, ar, asp = qc_check(cf, len(boxes), x1, y1, x2, y2, W, H)
    rec.update(conf=round(cf, 5), cls=cid, x1=x1, y1=y1, x2=x2, y2=y2,
               crop_w=x2 - x1, crop_h=y2 - y1,
               pad_l=pl, pad_t=pt, pad_r=pr, pad_b=pb,
               area_ratio=ar, aspect=asp, flags="|".join(flags), saved=False)
    return img[y1:y2, x1:x2], rec


def make_preview(img, rec, crop, max_h=PREVIEW_MAX_H):
    """원본+박스 | 크롭 을 가로로 이어붙인 검수 이미지"""
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    color = (0, 0, 255) if rec["flags"] else (0, 220, 0)
    th = max(2, min(vis.shape[:2]) // 250)
    # 마진 확장 박스(실제 크롭 범위)
    cv2.rectangle(vis, (rec["x1"], rec["y1"]), (rec["x2"], rec["y2"]), color, th)
    # 원본 검출 박스 (마진 전) — 하늘색
    cv2.rectangle(vis, (rec["x1"] + rec["pad_l"], rec["y1"] + rec["pad_t"]),
                       (rec["x2"] - rec["pad_r"], rec["y2"] - rec["pad_b"]),
                  (255, 200, 0), max(1, th // 2))

    cro = cv2.cvtColor(crop, cv2.COLOR_GRAY2BGR)
    s1, s2 = max_h / vis.shape[0], max_h / max(cro.shape[0], 1)
    vis = cv2.resize(vis, (int(vis.shape[1] * s1), max_h), interpolation=cv2.INTER_AREA)
    cro = cv2.resize(cro, (max(1, int(cro.shape[1] * s2)), max_h), interpolation=cv2.INTER_AREA)
    out = np.hstack([vis, np.zeros((max_h, 12, 3), np.uint8), cro])
    cv2.putText(out, f'{rec["image_id"]} conf={rec["conf"]:.2f} {rec["flags"] or "OK"}',
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    return out


# ─── 예행 ───────────────────────────────────────────────────────
probe = FILES[:6]
imgs  = [imread_u(p) for p in probe]
res   = det.predict([to_bgr(im) for im in imgs], conf=CONF_TH, iou=IOU_TH, imgsz=IMGSZ,
                    device=DEVICE, half=USE_HALF, verbose=False, max_det=10)

fig, axes = plt.subplots(2, len(probe), figsize=(3.3 * len(probe), 9.5))
prev = []
for j, (p, im, r) in enumerate(zip(probe, imgs, res)):
    crop, rec = crop_from(im, r, p.name); prev.append(rec)
    vis = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    cv2.rectangle(vis, (rec["x1"], rec["y1"]), (rec["x2"], rec["y2"]),
                  (255, 0, 0) if rec["flags"] else (0, 200, 0), max(3, im.shape[1] // 250))
    axes[0, j].imshow(vis[..., ::-1])
    axes[0, j].set_title(f'{p.stem}\nconf={rec["conf"]:.2f} {rec["flags"] or "OK"}', fontsize=9)
    out = finalize(crop)
    axes[1, j].imshow(out, cmap="gray")
    axes[1, j].set_title(f"저장될 크롭 {out.shape[1]}x{out.shape[0]}", fontsize=9)
    axes[0, j].axis("off"); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

display(pd.DataFrame(prev)[["image_id", "conf", "crop_w", "crop_h",
                            "pad_l", "pad_t", "pad_r", "pad_b", "flags"]])


## Part 6. 전체 배치 크롭 저장

**마진을 바꿨다면 반드시 `SKIP_EXISTING = False`로 두고 `images/` 폴더를 비우세요.**
안 그러면 옛 마진으로 만든 크롭과 섞여서, 눈에 안 보이는 채로 데이터셋이 오염됩니다.

RTX 5060 + FP16 기준 12,611장에 대략 8~15분입니다.


In [ ]:
# 마진 변경 후 재실행이면 True로 두고 실행 (images/ 전체 삭제)
CLEAN_BEFORE_RUN = False

if CLEAN_BEFORE_RUN:
    n_old = len(list(CROP_DIR.glob(f"*{SAVE_EXT}")))
    shutil.rmtree(CROP_DIR); CROP_DIR.mkdir(parents=True, exist_ok=True)
    print(f"기존 크롭 {n_old:,}장 삭제")

targets = FILES if LIMIT is None else FILES[:LIMIT]
if SKIP_EXISTING:
    targets = [p for p in targets if not (CROP_DIR / f"{p.stem}{SAVE_EXT}").exists()]
print(f"처리 대상 {len(targets):,}장 (전체 {len(FILES):,}장 중)")

records, errors = [], []

def _read(p):
    try:
        return p, imread_u(p), None
    except Exception as e:
        return p, None, f"{type(e).__name__}: {e}"

t0 = time.time(); n_saved = 0

with ThreadPoolExecutor(max_workers=READ_WORKERS) as pool, \
     tqdm(total=len(targets), desc="crop") as bar:

    for i in range(0, len(targets), BATCH):
        chunk  = targets[i:i + BATCH]
        loaded = list(pool.map(_read, chunk))

        ok = [(p, im) for p, im, err in loaded if im is not None]
        for p, im, err in loaded:
            if im is None:
                errors.append(f"{p.name}\tREAD\t{err}")
        if not ok:
            bar.update(len(chunk)); continue

        try:
            results = det.predict([to_bgr(im) for _, im in ok], conf=CONF_TH, iou=IOU_TH,
                                  imgsz=IMGSZ, device=DEVICE, half=USE_HALF,
                                  verbose=False, max_det=10)
        except Exception as e:
            for p, _ in ok:
                errors.append(f"{p.name}\tINFER\t{type(e).__name__}: {e}")
            bar.update(len(chunk)); continue

        for (p, im), r in zip(ok, results):
            try:
                crop, rec = crop_from(im, r, p.name)
                skip_save = ("no_det" in rec["flags"]) and (NO_DET_POLICY == "skip")
                if not skip_save and crop.size > 0:
                    out = finalize(crop)
                    imwrite_u(CROP_DIR / f"{p.stem}{SAVE_EXT}", out,
                              quality=JPG_QUALITY if SAVE_EXT.lower() in (".jpg", ".jpeg") else None)
                    rec["saved"] = True
                    rec["out_w"], rec["out_h"] = out.shape[1], out.shape[0]
                    n_saved += 1
                if SAVE_PREVIEW == "all" or (SAVE_PREVIEW == "flagged" and rec["flags"]):
                    imwrite_u(PREVIEW_DIR / f"{p.stem}.jpg", make_preview(im, rec, crop), quality=85)
                records.append(rec)
            except Exception as e:
                errors.append(f"{p.name}\tSAVE\t{type(e).__name__}: {e}")

        bar.update(len(chunk))

el = time.time() - t0
print(f"\n저장 {n_saved:,}장 / {el/60:.1f}분 ({n_saved/max(el,1e-9):.1f} img/s)")
print(f"예외 {len(errors)}건")
if errors:
    (BASE_OUT / "errors.txt").write_text("\n".join(errors), encoding="utf-8")
    for e in errors[:10]:
        print("  ", e)


## Part 7. 통계 + 누락 대조

나눠 실행했을 수 있으므로 **원본 폴더와 저장 폴더의 파일명 집합을 직접 비교**해 누락을 잡습니다.
로그만 믿으면 실행 단위로 쪼개져 전체 현황을 놓칩니다.

In [ ]:
log = pd.DataFrame(records)
if len(log):
    log.to_csv(BASE_OUT / "crop_log_v2.csv", index=False, encoding="utf-8-sig")
    fail = log[log.flags != ""]
    fail.to_csv(BASE_OUT / "crop_failed_v2.csv", index=False, encoding="utf-8-sig")

    print(f"이번 실행 {len(log):,}장 | 저장 {int(log.saved.sum()):,}장")
    print(f"QC 플래그 {len(fail):,}건 ({len(fail)/len(log)*100:.2f}%)")
    if len(fail):
        from collections import Counter
        for k, v in Counter(f for s in fail.flags for f in s.split("|")).most_common():
            print(f"   {k:14s} {v:6,}")

    print("\n[크롭 통계]")
    display(log[["conf", "crop_w", "crop_h", "pad_t", "pad_b", "area_ratio", "aspect"]]
            .describe().round(2))

    fig, ax = plt.subplots(1, 4, figsize=(19, 3.6))
    ax[0].hist(log.conf, bins=50, color="#3b7dd8"); ax[0].axvline(LOW_CONF, c="r", ls="--"); ax[0].set_title("confidence")
    ax[1].hist(log.area_ratio, bins=50, color="#3b7dd8")
    ax[1].axvline(MIN_AREA, c="r", ls="--"); ax[1].axvline(MAX_AREA, c="r", ls="--"); ax[1].set_title("area ratio")
    ax[2].hist(log.aspect, bins=50, color="#3b7dd8")
    ax[2].axvline(ASPECT_LO, c="r", ls="--"); ax[2].axvline(ASPECT_HI, c="r", ls="--"); ax[2].set_title("aspect (h/w)")
    ax[3].scatter(log.crop_w, log.crop_h, s=3, alpha=.25); ax[3].set_title("crop w vs h")
    plt.tight_layout(); plt.show()
else:
    print("이번 실행에서 새로 처리한 이미지가 없습니다 (전량 SKIP_EXISTING).")
    fail = pd.DataFrame()

saved_ids = {p.stem for p in CROP_DIR.glob(f"*{SAVE_EXT}")}
src_ids   = {p.stem for p in FILES}
missing   = sorted(src_ids - saved_ids)

print(f"\n원본 {len(src_ids):,} / 저장 {len(saved_ids):,} / 누락 {len(missing):,}")
if missing:
    (BASE_OUT / "missing_ids.txt").write_text("\n".join(missing), encoding="utf-8")
    print("  →", BASE_OUT / "missing_ids.txt")
    print("  예시:", missing[:10])


## Part 8. QC 케이스 검수

왼쪽 = 원본(초록/빨강: 실제 크롭 범위, 하늘색: 마진 전 검출 박스), 오른쪽 = 저장된 크롭.
`no_det` → `huge_box` → `low_conf` 순으로 심각합니다.

In [ ]:
SHOW_FLAG = None    # "no_det" / "huge_box" / "small_box" ... None이면 전체
SHOW_N    = 8

src = fail if len(fail) else pd.DataFrame()
if len(src) and SHOW_FLAG:
    src = src[src.flags.str.contains(SHOW_FLAG)]

if not len(src):
    print("표시할 QC 케이스가 없습니다.")
else:
    src = src.assign(prio=src.flags.apply(
            lambda f: 0 if "no_det" in f else 1 if "huge_box" in f else 2 if "low_conf" in f else 3)
         ).sort_values(["prio", "conf"]).head(SHOW_N)

    for _, r in src.iterrows():
        pv = PREVIEW_DIR / f"{r.image_id}.jpg"
        if pv.exists():
            img = imread_u(pv, cv2.IMREAD_COLOR)
        else:
            im = imread_u(SRC_DIR / r.file_name)
            img = make_preview(im, r.to_dict(), im[int(r.y1):int(r.y2), int(r.x1):int(r.x2)])
        plt.figure(figsize=(13, 6))
        plt.imshow(img[..., ::-1]); plt.axis("off")
        plt.title(f"{r.image_id} | conf={r.conf:.2f} | {r.flags}", color="crimson")
        plt.show()

    display(src[["image_id", "conf", "crop_w", "crop_h", "pad_t", "pad_b", "flags"]].round(3))


---
## 실행 순서

1. **Part 0~2** 실행
2. **Part 3** 잘림량 실측 (1~2분) → 권장 마진 확인
3. **Part 4** `USE_MEASURED=True`로 자동 반영 + 전/후 비교 확인
4. **Part 5** 예행 6장 — 손끝·손목이 모두 들어왔는지 최종 확인
5. **Part 6** `CLEAN_BEFORE_RUN=True`, `SKIP_EXISTING=False`로 전체 재실행
6. **Part 7~8** 누락 0건 + `no_det` 케이스 검수

## 판정 기준

| 항목 | 정상 | 벗어나면 |
|---|---|---|
| p95 잘림량 | ≤ 15px | 라벨링 기준이 흔들림 → Roboflow 라벨 재검토 |
| 검출 성공률 | ≥ 99.5% | `CONF_TH`를 0.15로 |
| conf 중앙값 | ≥ 0.90 | 실패 케이스 추가 라벨링 후 재학습 |
| aspect 중앙값 | 1.2~2.0 | 값↑ 전완 포함 / 값↓ 손가락 잘림 |

## 다음 단계

- **validation 1,425장 · test 200장도 동일 마진으로** 실행하세요. 스플릿마다 크롭 기준이 다르면 그 자체가 분포 이동이 되어 MAE에 잡힙니다. `SRC_DIR`·`BASE_OUT`만 바꾸고 Part 3은 건너뛴 채 Part 4에서 `USE_MEASURED=False` + 학습셋 마진값을 직접 입력하면 됩니다.
- 마진 값은 실험 기록에 남기세요. 논문 미명시 요소이므로 `PAPER_STRICT` 토글로 분리해 두는 편이 낫습니다.
